# L6: Building a Video Agent

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*non-text parts.*")
warnings.filterwarnings("ignore")

import json
import time
from google import genai
from google.genai import types
from google.genai.types import (
    GenerateVideosConfig,
    GenerateContentConfig,
)
from google.genai import types as genai_types
from IPython.display import (
    display,
    Video,
    Image as IPImage,
)

In [ ]:
import os
from helper import authenticate

credentials, project_id = authenticate()
client = genai.Client(
    project=project_id,
    location="global",
    credentials=credentials,
    http_options=types.HttpOptions(
         base_url=os.getenv("GOOGLE_VERTEX_BASE_URL")
    )
)

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>
</div>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different run results:</b> The output generated by AI models, and agents can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

In [ ]:
MODEL = "gemini-3.1-pro-preview"
IMAGE_MODEL = "gemini-3.1-flash-image-preview"
VIDEO_MODEL = "veo-3.1-fast-generate-001"

In [ ]:
VOICE_PROFILE = (
  """calm, clear, male voice with a neutral American accent.
     Warm baritone, steady pace around 140 words per minute.
     Professional and conversational, like a senior engineer explaining"
     to a colleague. No uptalk, no vocal fry. Even tone
     throughout, slight emphasis on key technical terms.
     Natural pauses between sentences."""
)

STYLE_PREFIX = (
  """Technical diagram on a dark navy background. Minimal flat design,
     sans-serif labels. 16:9 aspect ratio."""
)

## Tool 1: `plan_scenes`

In [ ]:
def plan_scenes(brief: str, num_scenes: int = 3) -> list:
    """Break a text brief into scene plans."""
    
    prompt = f"""
    Break this into {num_scenes} scenes for an explainer video.
    Brief: {brief}
    VOICE STYLE INSTRUCTIONS:
    Calm, clear, male voice with a neutral American accent. Warm baritone, 
    steady pace around 140 words per minute.
    Professional and conversational, like a senior engineer explaining to a 
    colleague. 
    No uptalk, no vocal fry. Even tone throughout, slight emphasis on key 
    technical terms. 
    Natural pauses between sentences.
    For each scene, return a JSON array where each element has:
    - visual_description: describe a clean diagram layout
    - narration_script: exactly ~20 words spoken in 8 seconds
    - camera_motion: one of [slow_left_to_right, slow_zoom_in, slow_zoom_out, static]
    Return ONLY the JSON array, no markdown.
    """.strip()

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
    )
    
    text = response.text.strip()
    
    # Clean up potential markdown formatting from the response
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]
    
    scenes = json.loads(text)
    
    for scene in scenes:
        scene["voice_profile"] = VOICE_PROFILE
        for key, val in scene.items():
            if isinstance(val, str):
                scene[key] = clean(val)
                
    return scenes

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Please wait:</b> This cell may take a minute to run.</p>

In [ ]:
from helper import clean
brief = (
    """Explain how RAG works -embedding the query, retrieving relevant documents,
    and generating a grounded answer."""
)

scene_plans = plan_scenes(brief)

In [ ]:
for i, scene in enumerate(scene_plans):
    print(f"\n--- Scene {i+1} ---")
    desc = scene['visual_description']
    print(f"Visual:    {desc[:70]}...")
    narr = scene['narration_script']
    print(f"Narration: {narr}")
    cam = scene['camera_motion']
    print(f"Camera:    {cam}")

## Tool 2: `generate_scene_image`

In [ ]:
from helper import extract_image
def generate_scene_image(
    visual_description: str,
    scene_num: int = 1,
) -> str:
    """Generate a reference frame. Returns file path."""
    response = client.models.generate_content(
        model=IMAGE_MODEL,
        contents=(
            STYLE_PREFIX
            + clean(visual_description)
        ),
        config=GenerateContentConfig(
            response_modalities=[
                "IMAGE", "TEXT"
            ],
        ),
    )
    img = extract_image(response)
    if img is None:
        raise ValueError(
            "No image returned"
        )
    path = f"scene_{scene_num}_ref.png"
    img.save(path)
    print("Reference frame for scene {scene_num}:")
    display(IPImage(filename=path, width=400))
    return path

## Tool 3: `generate_scene_video`

In [ ]:
def generate_scene_video(
    image_path: str,
    prompt: str,
    narration: str,
    voice_profile: str,
    scene_num: int = 1,
    timeout: int = 300,
) -> str:
    """Animate a reference frame into an 8s video with narration."""
    with open(image_path, "rb") as f:
        image_bytes = f.read()

    image_obj = genai_types.Image(
        image_bytes=image_bytes,
        mime_type="image/png",
    )

    full_prompt = (
        f"{clean(prompt)}\n\n"
        "Narration spoken in a"
        f" {clean(voice_profile)}:\n"
        f'\"{clean(narration)}\"'
    )

    operation = (
        client.models.generate_videos(
            model=VIDEO_MODEL,
            image=image_obj,
            prompt=full_prompt,
            config=GenerateVideosConfig(
                aspect_ratio="16:9",
                number_of_videos=1,
                duration_seconds=8,
                generate_audio=True,
            ),
        )
    )

    print(f"Operation: {operation.name}")
    start = time.time()

    while not operation.done:
        elapsed = int(
            time.time() - start
        )
        if elapsed > timeout:
            raise TimeoutError(
                "Veo timed out after"
                f" {timeout}s."
            )
        time.sleep(20)
        operation = (
            client.operations.get(operation)
        )
        print("  Processing..."f" ({elapsed}s elapsed)")

    video = (
        operation.result
        .generated_videos[0]
    )
    path = f"scene_{scene_num}.mp4"
    with open(path, "wb") as f:
        f.write(video.video.video_bytes)

    duration = int(time.time() - start)
    print(
        f"Scene {scene_num} video"
        f" ready ({duration}s):"
    )
    display(Video(path, embed=True, width=480, height=270))
    return path

## Tool 4: `evaluate_scene`

In [ ]:
def evaluate_scene(
    video_path: str,
    original_prompt: str,
    narration_script: str = "",
    threshold: float = 3.0,
) -> dict:
    """Score a video clip on multiple criteria using Gemini."""
    from google.genai.types import Part

    with open(video_path, "rb") as f:
        video_bytes = f.read()

    video_part = Part.from_bytes(
        data=video_bytes,
        mime_type="video/mp4",
    )

    if narration_script:
        narration_check = (
            "- narration_alignment: the spoken audio matches this expected narration:"
            f' \"{narration_script}\". Score 1 if the spoken words are completely'
            " different, incoherent, or unrecognizable."
        )
    else:
        narration_check = "- narration_sync: audio pacing matches visual flow"

    eval_prompt = (
        "Watch this 8-second video clip carefully.\n\n"
        f"It was generated from this prompt: {original_prompt}\n\n"
        "Score each criterion from 1 (poor) to 5 (excellent):\n"
        "- temporal_consistency: objects coherent across frames\n"
        "- motion_coherence: camera matches prompt direction\n"
        "- prompt_adherence: visuals match scene description\n"
        "- visual_quality: sharp, no artifacts or distortion\n"
        f"{narration_check}\n"
        "- voice_profile: voice tone and pace match professional narration\n"
        "- scene_continuity: visual style is consistent\n\n"
        f"Also classify the failure type if overall < {threshold}:\n"
        '- \"visual\" if the issue is with image quality or style\n'
        '- \"audio\" if the issue is with narration alignment or voice\n'
        '- \"none\" if the clip passes\n\n'
        "Return ONLY a JSON object:\n"
        "{\n"
        '  \"scores\": {\"criterion_name\": score},\n'
        '  \"overall\": <average>,\n'
        f'  \"pass\": <true if overall >= {threshold}>,\n'
        '  \"failure_type\": \"visual\" | \"audio\" | \"none\",\n'
        '  \"feedback\": \"<improvements if fail, else OK>\"\n'
        "}"
    )

    response = client.models.generate_content(
        model=MODEL,
        contents=[video_part, eval_prompt],
    )

    text = response.text.strip()
    if not text:
        return {
            "overall": 0,
            "pass": False,
            "failure_type": "audio",
            "feedback": "Empty response",
            "scores": {},
        }

    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]

    result = json.loads(text)
    result["pass"] = result["overall"] >= threshold

    narr_score = result["scores"].get("narration_alignment", result["scores"].get("narration_sync", 5))
    if narr_score < 3:
        result["pass"] = False
        result["failure_type"] = "audio"
        result["feedback"] = result["feedback"] + " Narration does not match the expected script."

    overall = result['overall']
    print(f"   Overall:      {overall:.1f}")
    passed = result['pass']
    print(f"   Pass:         {passed}")
    ftype = result['failure_type']
    print(f"   Failure type: {ftype}")
    fb = result['feedback']
    print(f"   Feedback:     {fb}")

    if not result['pass']:
        step = 'image' if ftype == 'visual' else 'video'
        print(f"   -> Agent will retry from {step} step")

    return result

## Wiring the Agent

In [ ]:
CORRUPTED_NARRATION = (
    """The quantum blockchain leverages distributed photon matrices to 
    synergize cross-dimensional neural pathways."""
)

print("Corrupted narration for scene 1:")
print(CORRUPTED_NARRATION)
print()
print("Real narration for scene 1:")
print(
    scene_plans[0]["narration_script"]
)

In [ ]:
from helper import make_display_tool
from google.adk import Agent

plan_scenes_tool = make_display_tool(
    plan_scenes
)
generate_scene_image_tool = (
    make_display_tool(generate_scene_image)
)
generate_scene_video_tool = (
    make_display_tool(generate_scene_video)
)
evaluate_scene_tool = make_display_tool(
    evaluate_scene
)

PLANS_JSON = json.dumps(
    scene_plans, indent=2
)
CORRUPT = CORRUPTED_NARRATION

In [ ]:
SYSTEM_PROMPT = (
    "You are a video production agent.\n"
    "Your goal is to produce a high-quality video for every scene in the plan below.\n\n"
    "Scene plans:\n"
    f"{PLANS_JSON}\n\n"
    "You have three tools: generate_scene_image, generate_scene_video, and evaluate_scene.\n\n"
    "Process ONE scene at a time. Do NOT batch. Always evaluate after generating a video."
    " Do not move to the next scene until the current one passes.\n\n"
    "For scene 1 only, on the FIRST generate_scene_video call, use this corrupted audio"
    " instead of the real one:\n"
    f'"{CORRUPT}"\n'
    "On any retry of scene 1, use the real narration_script from the plan.\n\n"
    "For all other scenes, always use the real narration_script.\n\n"
    "When calling evaluate_scene, always pass the REAL narration_script from the scene plan.\n\n"
    "If evaluation fails:\n"
    "- failure_type 'audio': regenerate the video only, reuse the same image.\n"
    "- failure_type 'visual': regenerate both the image and the video.\n\n"
    "Maximum 1 retry per scene. After all scenes pass, list the video file paths."
)

In [ ]:
from google.adk.models import Gemini
class DLAIGemini(Gemini):
    _client: genai.Client = None  # class-level cache
    
    @property
    def api_client(self) -> genai.Client:
        if self._client is None:
            self._client = genai.Client(
                project=project_id,
                location="global",
                credentials=credentials,
                http_options=types.HttpOptions(
                    base_url=os.getenv("GOOGLE_VERTEX_BASE_URL")
                )
            )
        return self._client
        
agent = Agent(
    name="video_agent",
    model=DLAIGemini(model=MODEL),
    tools=[
        generate_scene_image_tool,
        generate_scene_video_tool,
        evaluate_scene_tool,
    ],
    instruction=SYSTEM_PROMPT,
)

print("Agent ready.")

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import (
    InMemorySessionService,
)

session_service = InMemorySessionService()
runner = Runner(
    agent=agent,
    app_name="video_agent",
    session_service=session_service,
)

session = await session_service.create_session(
    app_name="video_agent",
    user_id="user",
)

message = genai_types.Content(
    role="user",
    parts=[
        genai_types.Part(
            text="Start video production."
        )
    ],
)

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Please wait:</b> This cell may take a few minutes to run.</p>

In [ ]:
print("Running agent - ~10 min for 3 scenes...")
print("Scene 1 will generate with wrong narration. Listen for the bad audio, then watch the agent fix it.\n")

async for event in runner.run_async(
    user_id="user",
    session_id=session.id,
    new_message=message,
):
    if event.is_final_response():
        print("\n--- Agent complete ---")
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text)

## Final Step: Concatenate with ffmpeg

In [ ]:
with open("scenes.txt", "w") as f:
    f.write("file 'scene_1.mp4'\n")
    f.write("file 'scene_2.mp4'\n")
    f.write("file 'scene_3.mp4'\n")

!ffmpeg -f concat -safe 0 -i scenes.txt -c copy rag_explainer.mp4 -y -loglevel error

print("Final video: rag_explainer.mp4")

display(Video("rag_explainer.mp4", embed=True, width=800, height=450))

**Resources**
- [Agent Development Kit](https://google.github.io/adk-docs/)
- [Veo documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/video/overview)